# BLIP-2 VQA Fusion Experiment — Analysis Notebook

This notebook walks through:
1. **Dataset exploration** – question-type and answer distributions
2. **Model comparison** – accuracy curves for all fusion variants
3. **Error analysis** – where the models fail and why
4. **Attention visualisation** – Q-Former cross-attention heat-maps

In [ ]:
import sys
sys.path.insert(0, '..')  # project root

import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

import torch
from PIL import Image

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1  Dataset Exploration

In [ ]:
# ── Adjust these paths to your local dataset location ──────────────────────
VAL_QUESTIONS = '../data/raw/v2_OpenEnded_mscoco_val2014_questions.json'
VAL_ANNOTATIONS = '../data/raw/v2_mscoco_val2014_annotations.json'
# ────────────────────────────────────────────────────────────────────────────

with open(VAL_QUESTIONS) as f:
    q_data = json.load(f)
with open(VAL_ANNOTATIONS) as f:
    a_data = json.load(f)

questions = q_data['questions']
annotations = {a['question_id']: a for a in a_data['annotations']}
print(f'Questions: {len(questions):,}')

In [ ]:
# Answer-type distribution
answer_types = Counter(
    annotations[q['question_id']]['answer_type']
    for q in questions
    if q['question_id'] in annotations
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].pie(
    answer_types.values(),
    labels=answer_types.keys(),
    autopct='%1.1f%%',
    startangle=90,
)
axes[0].set_title('Answer-type distribution')

# Top-20 most frequent answers
top_answers = Counter(
    annotations[q['question_id']]['multiple_choice_answer']
    for q in questions
    if q['question_id'] in annotations
).most_common(20)

labels, counts = zip(*top_answers)
axes[1].barh(labels[::-1], counts[::-1], color='steelblue')
axes[1].set_title('Top-20 most frequent answers')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.savefig('answer_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Question length distribution
q_lengths = [len(q['question'].split()) for q in questions]

plt.figure(figsize=(8, 4))
plt.hist(q_lengths, bins=range(1, 30), edgecolor='white', color='coral')
plt.xlabel('Question length (words)')
plt.ylabel('Count')
plt.title('Distribution of question lengths')
plt.tight_layout()
plt.savefig('question_length_distribution.png', bbox_inches='tight')
plt.show()
print(f'Mean: {np.mean(q_lengths):.1f}  |  Median: {np.median(q_lengths):.1f}  |  Max: {max(q_lengths)}')

## 2  Model Comparison

In [ ]:
# ── Paste your results here ─────────────────────────────────────────────────
# Format: model_name → {split: accuracy}
results = {
    'BLIP-2 Q-Former':   {'val': 0.0},
    'Concat Fusion':     {'val': 0.0},
    'Bilinear Fusion':   {'val': 0.0},
    'Attention Fusion':  {'val': 0.0},
    'MLB Fusion':        {'val': 0.0},
}
# ────────────────────────────────────────────────────────────────────────────

models = list(results.keys())
val_accs = [results[m]['val'] for m in models]

plt.figure(figsize=(9, 4))
bars = plt.barh(models[::-1], val_accs[::-1], color='mediumseagreen')
plt.xlabel('VQA Accuracy (%)')
plt.title('Model comparison on VQAv2 val')
for bar, val in zip(bars, val_accs[::-1]):
    plt.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
             f'{val:.2f}', va='center')
plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

## 3  Error Analysis

In [ ]:
from data.vqa_dataset import normalize_answer

PREDICTIONS_FILE = '../results/val_predictions.json'  # produced by evaluate.py

try:
    with open(PREDICTIONS_FILE) as f:
        preds = json.load(f)   # [{question_id, answer}, ...]

    errors, correct = [], []
    for p in preds:
        qid = int(p['question_id'])
        if qid not in annotations:
            continue
        ann = annotations[qid]
        human_answers = [normalize_answer(a['answer']) for a in ann['answers']]
        acc = min(human_answers.count(normalize_answer(p['answer'])) / 3.0, 1.0)
        if acc == 0:
            errors.append({'question_id': qid, 'predicted': p['answer'],
                           'gold': ann['multiple_choice_answer'],
                           'type': ann['answer_type']})
        else:
            correct.append(qid)

    print(f'Correct: {len(correct):,}  |  Errors: {len(errors):,}')

    # Error type breakdown
    error_types = Counter(e['type'] for e in errors)
    plt.figure(figsize=(6, 3))
    plt.bar(error_types.keys(), error_types.values(), color='salmon')
    plt.title('Error distribution by answer type')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.savefig('error_analysis.png', bbox_inches='tight')
    plt.show()

    # Sample errors
    print('\nSample errors (first 5):')
    for e in errors[:5]:
        q_text = next((q['question'] for q in questions if q['question_id'] == e['question_id']), '?')
        print(f"  Q: {q_text}")
        print(f"    Predicted: {e['predicted']}  |  Gold: {e['gold']}")
        print()
except FileNotFoundError:
    print(f'Predictions file not found: {PREDICTIONS_FILE}\nRun evaluate.py first.')

## 4  Q-Former Attention Visualisation

In [ ]:
from torchvision import transforms
from models.qformer import QFormer, QFormerConfig

def show_attention(image_path: str, query_idx: int = 0) -> None:
    """Visualise cross-attention weights of a Q-Former query token."""
    config = QFormerConfig(num_query_tokens=32, vision_width=768, hidden_size=256,
                           num_hidden_layers=4, num_attention_heads=4, intermediate_size=1024)
    qformer = QFormer(config).eval()

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    img = Image.open(image_path).convert('RGB')
    img_t = transform(img).unsqueeze(0)              # [1, 3, 224, 224]

    # Dummy visual features (14x14 patches for a 224-px image)
    num_patches = 14 * 14
    vis_feats = torch.randn(1, num_patches, config.vision_width)

    # Monkey-patch to capture attention weights from the first cross-attn layer
    attn_weights = {}

    original_forward = qformer.layers[0].cross_attn.forward
    def hook_forward(q, k, v, mask=None):
        B, Nq, _ = q.shape
        Nkv = k.shape[1]
        heads = qformer.layers[0].cross_attn.num_heads
        head_dim = qformer.layers[0].cross_attn.head_dim
        scale = head_dim ** -0.5
        qp = qformer.layers[0].cross_attn.q_proj(q).view(B, Nq, heads, head_dim).transpose(1, 2)
        kp = qformer.layers[0].cross_attn.k_proj(k).view(B, Nkv, heads, head_dim).transpose(1, 2)
        vp = qformer.layers[0].cross_attn.v_proj(v).view(B, Nkv, heads, head_dim).transpose(1, 2)
        a = torch.softmax(torch.matmul(qp, kp.transpose(-2, -1)) * scale, dim=-1)
        attn_weights['w'] = a.detach()
        out = torch.matmul(a, vp).transpose(1, 2).reshape(B, Nq, -1)
        return qformer.layers[0].cross_attn.out_proj(out)
    qformer.layers[0].cross_attn.forward = hook_forward

    with torch.no_grad():
        qformer(vis_feats)

    # Average over heads
    weights = attn_weights['w'][0].mean(0)           # [Nq, num_patches]
    attn_map = weights[query_idx].reshape(14, 14).numpy()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(img)
    axes[0].axis('off')
    axes[0].set_title('Input image')
    im = axes[1].imshow(attn_map, cmap='hot', interpolation='bilinear')
    axes[1].axis('off')
    axes[1].set_title(f'Cross-attention (query {query_idx})')
    plt.colorbar(im, ax=axes[1])
    plt.tight_layout()
    plt.savefig(f'attention_query_{query_idx}.png', bbox_inches='tight')
    plt.show()

# Uncomment and set a real image path to run:
# show_attention('/path/to/image.jpg', query_idx=0)